# Práctica 1: Exploración de LLMs con API y LangChain

## Objetivos
- Aplicar los conceptos de conexión API directa y con LangChain
- Experimentar con diferentes parámetros (temperature, max_tokens, modelos)
- Implementar streaming y memoria conversacional
- Resolver ejercicios prácticos de forma autónoma

## Instrucciones
Completa los ejercicios en orden. Cada sección tiene celdas de código para que implementes las soluciones.

## Configuración Inicial

In [ ]:
import os

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

MODELO = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
MODELO_RAPIDO = os.getenv("GROQ_MODEL_FAST", "llama-3.1-8b-instant")

from groq import Groq

client = Groq()  # lee GROQ_API_KEY del entorno
print("✅ Cliente configurado correctamente")

---
## Ejercicio 1: Comparación de Modelos

**Instrucción:** Prueba al menos 3 modelos diferentes de Groq (`llama-3.1-8b-instant`,
`llama-3.3-70b-versatile`, `openai/gpt-oss-20b`) con el mismo prompt.
Observa las diferencias en velocidad, calidad y cantidad de tokens usados.

> El catálogo vigente está en [console.groq.com/docs/models](https://console.groq.com/docs/models).
> Si alguno de esos identificadores ya no existe, reemplázalo por otro del catálogo.
> Recuerda que la capa gratuita permite ~30 peticiones por minuto: no ejecutes esta celda en bucle.

In [ ]:
# Escribe aquí tu código para probar diferentes modelos
modelos = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile", "openai/gpt-oss-20b"]
prompt = "Explica la diferencia entre IA débil y IA fuerte en 3 oraciones."

for modelo in modelos:
    print(f"\n{'='*50}")
    print(f"Modelo: {modelo}")
    print('='*50)
    response = client.chat.completions.create(
        model=modelo,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=200
    )
    print(f"Respuesta: {response.choices[0].message.content}")
    print(f"Tokens: {response.usage.total_tokens}")

---
## Ejercicio 2: Efecto de la Temperatura

**Instrucción:** Usa el mismo modelo y prompt con diferentes valores de temperature (0.0, 0.5, 1.0, 1.5). 
Ejecuta cada uno 3 veces y analiza la variabilidad de las respuestas.

In [ ]:
# Escribe aquí tu código para experimentar con temperature
temperaturas = [0.0, 0.5, 1.0, 1.5]
prompt = "Inventa un nombre creativo para un asistente de IA educativo."

for temp in temperaturas:
    print(f"\n--- Temperature = {temp} ---")
    for i in range(3):
        response = client.chat.completions.create(
            model=MODELO_RAPIDO,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=50
        )
        print(f"  Intento {i+1}: {response.choices[0].message.content}")

---
## Ejercicio 3: Sistema vs Usuario

**Instrucción:** Crea 3 system prompts diferentes para un mismo tema y compara cómo cambia el 
comportamiento del modelo. Por ejemplo: un tutor estricto, uno amigable y uno sarcástico.

In [ ]:
# Escribe aquí tu código para probar system prompts
system_prompts = [
    "Eres un tutor de programación estricto y exigente. Corriges cada error.",
    "Eres un tutor de programación muy paciente y alentador. Usas emojis.",
    "Eres un tutor de programación con humor sarcástico, pero enseñas bien."
]

pregunta = "¿Qué es una variable en Python?"

for sp in system_prompts:
    print(f"\n{'='*50}")
    print(f"System: {sp[:40]}...")
    print('='*50)
    response = client.chat.completions.create(
        model=MODELO_RAPIDO,
        messages=[
            {"role": "system", "content": sp},
            {"role": "user", "content": pregunta}
        ],
        temperature=0.7,
        max_tokens=150
    )
    print(f"Respuesta: {response.choices[0].message.content}")

---
## Ejercicio 4: Streaming en Tiempo Real

**Instrucción:** Implementa una llamada con streaming usando el cliente de Groq directo. 
Muestra cada chunk a medida que llega.

In [ ]:
# Escribe aquí tu código para streaming
print("Respuesta en streaming:")
print("-" * 40)

stream = client.chat.completions.create(
    model=MODELO_RAPIDO,
    messages=[{"role": "user", "content": "Explícame qué es el streaming en API en 2 oraciones."}],
    temperature=0.3,
    max_tokens=100,
    stream=True
)

respuesta_completa = ""
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        contenido = chunk.choices[0].delta.content
        print(contenido, end="", flush=True)
        respuesta_completa += contenido

print("\n" + "-" * 40)
print(f"\nTotal caracteres: {len(respuesta_completa)}")

---
## Ejercicio 5: Chatbot con Memoria (Desafío)

**Instrucción:** Implementa un chatbot simple que mantenga el historial de conversación en una lista 
de mensajes. Debe recordar el nombre del usuario y responder preguntas de seguimiento.

**Pistas:**
- Usa una lista `messages` que incluya system, user y assistant roles
- Después de cada respuesta, agregala a la lista
- Para cada nuevo mensaje, envía toda la lista al modelo

In [ ]:
# Escribe aquí tu chatbot con memoria
messages = [
    {"role": "system", "content": "Eres un asistente amigable. Recuerdas la conversación."}
]

# Simula una conversación
preguntas = [
    "Hola, me llamo Ana.",
    "¿Cómo me llamo?",
    "¿De qué hablamos recién?"
]

for pregunta in preguntas:
    print(f"\nUsuario: {pregunta}")
    messages.append({"role": "user", "content": pregunta})

    response = client.chat.completions.create(
        model=MODELO_RAPIDO,
        messages=messages,
        temperature=0.3,
        max_tokens=100
    )

    respuesta = response.choices[0].message.content
    print(f"Asistente: {respuesta}")
    messages.append({"role": "assistant", "content": respuesta})

print("\n" + "=" * 40)
print(f"Historial completo: {len(messages)} mensajes")

---
## Entregable

Completa todos los ejercicios y documenta:
1. ¿Qué modelo de Groq te gustó más y por qué?
2. ¿Cómo afecta la temperature a las respuestas?
3. ¿Qué ventaja tiene el streaming?
4. ¿Por qué es importante la memoria en un chatbot?

Responde en una celda markdown abajo.

### Tus respuestas:

1. 
2. 
3. 
4. 